# 유니버스 2015+ 과거 데이터 수집 (pykrx Primary · yfinance Fallback)

- **Primary**: `pykrx.stock.get_market_ohlcv_by_date` (KRX 공식 · 액면분할/배당 자동 반영)
- **Fallback**: `yfinance` `{code}.KS` (pykrx 실패 종목만 재시도)
- **KOSPI 지수**: `pykrx` 코드 `1001`
- **VIX**: `yfinance ^VIX`
- **기간**: 2015-01-01 ~ 오늘

### 알려진 한계
2015년 당시 비상장/액면분할 전 종목은 해당 시점 데이터가 없어 자동으로 시작일부터 수집됩니다. 완벽한 point-in-time 재현(상폐 종목 포함)은 MVP 범위 밖이며 생존편향 일부를 수용합니다.

## 1. 환경 설정

In [ ]:
!pip install -q pykrx yfinance pandas pyarrow tqdm

In [ ]:
import os
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from pykrx import stock as krx
import yfinance as yf

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 20)

START_DATE = "20150101"
END_DATE   = datetime.now().strftime("%Y%m%d")
OUTPUT_DIR = Path("./data")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"수집 기간: {START_DATE} ~ {END_DATE}")
print(f"출력 경로: {OUTPUT_DIR.resolve()}")

## 2. 유니버스 (한국 대형주 137종목)

In [ ]:
UNIVERSE = [
    ("005930", "삼성전자"),       ("000660", "SK하이닉스"),
    ("373220", "LG에너지솔루션"), ("207940", "삼성바이오로직스"),
    ("005380", "현대차"),          ("000270", "기아"),
    ("051910", "LG화학"),          ("006400", "삼성SDI"),
    ("035420", "NAVER"),           ("105560", "KB금융"),
    ("055550", "신한지주"),        ("012330", "현대모비스"),
    ("028260", "삼성물산"),        ("003550", "LG"),
    ("086790", "하나금융지주"),    ("034730", "SK"),
    ("096770", "SK이노베이션"),    ("017670", "SK텔레콤"),
    ("316140", "우리금융지주"),    ("032830", "삼성생명"),
    ("000810", "삼성화재"),        ("003670", "포스코홀딩스"),
    ("066570", "LG전자"),          ("030200", "KT"),
    ("015760", "한국전력"),        ("035720", "카카오"),
    ("012450", "한화에어로스페이스"), ("009150", "삼성전기"),
    ("032640", "LG유플러스"),      ("090430", "아모레퍼시픽"),
    ("010950", "S-Oil"),            ("003490", "대한항공"),
    ("011170", "롯데케미칼"),      ("047810", "한국항공우주"),
    ("042660", "한화오션"),        ("034020", "두산에너빌리티"),
    ("259960", "크래프톤"),        ("036570", "엔씨소프트"),
    ("021240", "코웨이"),          ("128940", "한미약품"),
    ("000100", "유한양행"),        ("068270", "셀트리온"),
    ("018260", "삼성에스디에스"),  ("010130", "고려아연"),
    ("004020", "현대제철"),        ("073240", "금호석유"),
    ("024110", "IBK기업은행"),     ("000720", "현대건설"),
    ("011070", "LG이노텍"),        ("004170", "신세계"),
    ("047050", "포스코인터내셔널"), ("001450", "현대해상"),
    ("078930", "GS"),               ("036460", "한국가스공사"),
    ("010140", "삼성중공업"),      ("033780", "KT&G"),
    ("006800", "미래에셋증권"),    ("352820", "하이브"),
    ("267250", "HD현대"),          ("329180", "HD현대중공업"),
    ("241560", "두산밥캣"),        ("338220", "에코프로비엠"),
    ("247540", "에코프로"),        ("282330", "BGF리테일"),
    ("007070", "GS리테일"),        ("003230", "삼양식품"),
    ("271560", "오리온"),          ("030000", "제일기획"),
    ("000150", "두산"),            ("006360", "GS건설"),
    ("326030", "SK바이오팜"),      ("302440", "SK바이오사이언스"),
    ("011790", "SKC"),              ("029780", "삼성카드"),
    ("005870", "한화생명"),        ("016360", "삼성증권"),
    ("005830", "DB손해보험"),      ("000080", "하이트진로"),
    ("161390", "한국타이어"),      ("004990", "롯데지주"),
    ("010060", "OCI"),              ("042670", "HD현대인프라코어"),
    ("004000", "롯데정밀화학"),    ("000990", "DB하이텍"),
    ("023530", "롯데쇼핑"),        ("002790", "아모레G"),
    ("009830", "한화솔루션"),      ("001120", "LX인터내셔널"),
    ("251270", "넷마블"),          ("041510", "에스엠"),
    ("035900", "JYP Ent"),          ("018880", "한온시스템"),
    ("139480", "이마트"),          ("011200", "HMM"),
    ("009540", "한진칼"),          ("034230", "파라다이스"),
    ("003380", "하림지주"),
    ("097950", "CJ제일제당"),      ("000120", "CJ대한통운"),
    ("001040", "CJ"),               ("005300", "롯데칠성"),
    ("280360", "롯데웰푸드"),      ("004370", "농심"),
    ("007310", "오뚜기"),          ("008770", "호텔신라"),
    ("000215", "DL이앤씨"),        ("000210", "DL"),
    ("138040", "메리츠금융지주"),  ("000060", "메리츠화재"),
    ("005940", "NH투자증권"),      ("071050", "한국금융지주"),
    ("039490", "키움증권"),        ("377300", "카카오페이"),
    ("293490", "카카오뱅크"),      ("000880", "한화"),
    ("272210", "한화시스템"),      ("079550", "LIG넥스원"),
    ("010120", "LS ELECTRIC"),      ("006260", "LS"),
    ("298040", "효성중공업"),      ("298000", "효성티앤씨"),
    ("004800", "효성"),            ("022100", "포스코DX"),
    ("000070", "삼양홀딩스"),      ("192820", "코스맥스"),
    ("051600", "한전KPS"),         ("028050", "삼성엔지니어링"),
    ("012750", "에스원"),          ("001740", "SK네트웍스"),
    ("007700", "F&F"),              ("035250", "강원랜드"),
    ("069960", "현대백화점"),      ("057050", "현대홈쇼핑"),
    ("014680", "한솔케미칼"),      ("000370", "한화손해보험"),
    ("006120", "SK디스커버리"),    ("001230", "동국제강"),
    ("017800", "현대엘리베이터"),  ("001680", "대상"),
    ("032350", "롯데렌탈"),        ("114090", "GKL"),
    ("267270", "현대건설기계"),    ("111770", "영원무역"),
    ("009970", "영원무역홀딩스"),  ("004490", "세방전지"),
    ("009240", "한샘"),            ("192400", "쿠쿠홀딩스"),
    ("204320", "만도"),            ("012630", "HDC"),
    ("294870", "HDC현대산업개발"), ("005960", "동원F&B"),
    ("008560", "메리츠증권"),      ("010780", "아이에스동서"),
    ("002350", "넥센타이어"),      ("002410", "한진"),
    ("003690", "코리안리"),        ("005610", "SPC삼립"),
    ("008930", "한미사이언스"),    ("006650", "대한유화"),
    ("000240", "한국앤컴퍼니"),    ("014820", "동원시스템즈"),
    ("001800", "오리온홀딩스"),    ("005490", "POSCO"),
    ("079960", "동양생명"),
]

NAME_MAP = dict(UNIVERSE)
print(f"유니버스: {len(UNIVERSE)}종목")

## 3. 수집 함수 — pykrx Primary, yfinance Fallback

- pykrx는 기본적으로 수정주가(액면분할·배당 반영)를 반환
- 네트워크/rate limit 오류 시 지수 백오프로 최대 3회 재시도
- 빈 DataFrame은 "해당 시점 미상장"으로 간주하고 실패로 카운트하지 않음

In [ ]:
PYKRX_RENAME = {
    "시가": "Open", "고가": "High", "저가": "Low",
    "종가": "Close", "거래량": "Volume",
    "거래대금": "TradeValue", "등락률": "PctChange",
}
OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]


def fetch_pykrx(ticker: str, start: str, end: str, retries: int = 3):
    """KRX 공식 데이터. 성공=DataFrame, 미상장/데이터없음=빈 DataFrame, 오류=None."""
    last_err = None
    for attempt in range(retries):
        try:
            df = krx.get_market_ohlcv_by_date(start, end, ticker, adjusted=True)
            if df is None or df.empty:
                return pd.DataFrame()
            df = df.rename(columns=PYKRX_RENAME)
            missing = [c for c in OHLCV_COLS if c not in df.columns]
            if missing:
                raise KeyError(f"pykrx 컬럼 누락: {missing}, got={list(df.columns)}")
            df.index = pd.to_datetime(df.index)
            df.index.name = "Date"
            return df[OHLCV_COLS]
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    print(f"  [pykrx 실패] {ticker}: {last_err}")
    return None


def fetch_yfinance(ticker: str, start: str, end: str, retries: int = 3):
    """Fallback: yfinance .KS. 네트워크 오류만 재시도."""
    yf_symbol = f"{ticker}.KS"
    start_fmt = f"{start[:4]}-{start[4:6]}-{start[6:]}"
    end_fmt   = f"{end[:4]}-{end[4:6]}-{end[6:]}"
    last_err = None
    for attempt in range(retries):
        try:
            df = yf.download(yf_symbol, start=start_fmt, end=end_fmt,
                             progress=False, auto_adjust=True, threads=False)
            if df is None or df.empty:
                return pd.DataFrame()
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.index = pd.to_datetime(df.index)
            df.index.name = "Date"
            missing = [c for c in OHLCV_COLS if c not in df.columns]
            if missing:
                raise KeyError(f"yfinance 컬럼 누락: {missing}, got={list(df.columns)}")
            return df[OHLCV_COLS]
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    print(f"  [yfinance 실패] {yf_symbol}: {last_err}")
    return None


def fetch_ohlcv(ticker: str, start: str, end: str):
    """Primary → Fallback 순서로 시도. (df, source) 반환."""
    df = fetch_pykrx(ticker, start, end)
    if df is not None and not df.empty:
        return df, "pykrx"
    df_yf = fetch_yfinance(ticker, start, end)
    if df_yf is not None and not df_yf.empty:
        return df_yf, "yfinance"
    if df is not None and df.empty and df_yf is not None and df_yf.empty:
        return pd.DataFrame(), "empty"
    return None, "failed"


## 4. 유니버스 일괄 수집

In [ ]:
results: dict[str, pd.DataFrame] = {}
source_log: list[dict] = []

for ticker, name in tqdm(UNIVERSE, desc="유니버스 수집"):
    df, source = fetch_ohlcv(ticker, START_DATE, END_DATE)
    first = df.index[0].date().isoformat() if df is not None and not df.empty else None
    last  = df.index[-1].date().isoformat() if df is not None and not df.empty else None
    rows  = len(df) if df is not None else 0
    source_log.append({
        "ticker": ticker, "name": name, "source": source,
        "rows": rows, "first": first, "last": last,
    })
    if df is not None and not df.empty:
        results[ticker] = df
    time.sleep(0.1)  # KRX 서버 부담 완화

source_df = pd.DataFrame(source_log)
summary = source_df["source"].value_counts().to_dict()
print("\n수집 요약:")
for k, v in summary.items():
    print(f"  {k:10s}: {v:3d}")

In [ ]:
# yfinance로 fallback된 종목
fb = source_df[source_df["source"] == "yfinance"]
if not fb.empty:
    print("yfinance fallback 사용 종목:")
    print(fb[["ticker", "name", "rows", "first", "last"]].to_string(index=False))
else:
    print("전 종목 pykrx로 수집 성공 (fallback 0건)")

In [ ]:
# 수집 실패 종목 (양쪽 모두 실패)
failed = source_df[source_df["source"] == "failed"]
if not failed.empty:
    print("수집 실패 (재실행 대상):")
    print(failed[["ticker", "name"]].to_string(index=False))
else:
    print("완전 실패 0건")

# 2015년 이후 상장 (survival bias 확인용)
source_df["first_dt"] = pd.to_datetime(source_df["first"])
late_listed = source_df[source_df["first_dt"] > "2015-01-31"].sort_values("first_dt")
if not late_listed.empty:
    print(f"\n2015년 2월 이후 상장/데이터 시작 ({len(late_listed)}종목):")
    print(late_listed[["ticker", "name", "first", "rows"]].to_string(index=False))

## 5. KOSPI 지수 (pykrx · 코드 1001)

In [ ]:
kospi_raw = krx.get_index_ohlcv_by_date(START_DATE, END_DATE, "1001")
kospi = kospi_raw.rename(columns={
    "시가": "Open", "고가": "High", "저가": "Low",
    "종가": "Close", "거래량": "Volume",
    "거래대금": "TradeValue", "상장시가총액": "MarketCap",
})
kospi.index = pd.to_datetime(kospi.index)
kospi.index.name = "Date"

print(f"KOSPI: {kospi.index[0].date()} ~ {kospi.index[-1].date()} ({len(kospi)} rows)")
kospi.tail()

## 6. VIX (yfinance · ^VIX)

In [ ]:
vix_raw = yf.download("^VIX",
                      start=f"{START_DATE[:4]}-{START_DATE[4:6]}-{START_DATE[6:]}",
                      end=f"{END_DATE[:4]}-{END_DATE[4:6]}-{END_DATE[6:]}",
                      progress=False, auto_adjust=False, threads=False)
if isinstance(vix_raw.columns, pd.MultiIndex):
    vix_raw.columns = vix_raw.columns.get_level_values(0)
vix = vix_raw[["Open", "High", "Low", "Close"]].copy()
vix.index = pd.to_datetime(vix.index)
vix.index.name = "Date"

print(f"VIX: {vix.index[0].date()} ~ {vix.index[-1].date()} ({len(vix)} rows)")
vix.tail()

## 7. 저장 (Parquet)

In [ ]:
# 패널 구조 (ticker, field) 멀티컬럼
panel = pd.concat(results, axis=1, names=["ticker", "field"])

panel.to_parquet(OUTPUT_DIR / "universe_ohlcv.parquet")
kospi.to_parquet(OUTPUT_DIR / "kospi_1001.parquet")
vix.to_parquet(OUTPUT_DIR / "vix.parquet")
source_df.drop(columns=["first_dt"], errors="ignore").to_csv(
    OUTPUT_DIR / "source_log.csv", index=False, encoding="utf-8-sig"
)

for p in sorted(OUTPUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:32s} {size_kb:>10.1f} KB")

## 8. Sanity Check

In [ ]:
# 삼성전자로 기본 검증
samsung = results["005930"]
print(f"삼성전자 (005930)")
print(f"  기간     : {samsung.index[0].date()} ~ {samsung.index[-1].date()}")
print(f"  거래일수 : {len(samsung):,}")
print(f"  최근 종가: {samsung['Close'].iloc[-1]:,.0f}")
print(f"  최고가   : {samsung['High'].max():,.0f}  ({samsung['High'].idxmax().date()})")
print(f"  최저가   : {samsung['Low'].min():,.0f}  ({samsung['Low'].idxmin().date()})")

In [ ]:
# Colab에서 다운로드하려면 아래 주석 해제
# from google.colab import files
# for p in OUTPUT_DIR.iterdir():
#     files.download(str(p))